# Fruit Freshness Classification — Course-Scope Version

This notebook intentionally stays within the methods covered in the uploaded course slides.

Models:
1. Fully connected neural network baseline
2. Convolutional neural network (CNN)
3. CNN with simple data augmentation

Deep-learning methods used:
- Fully connected layers
- ReLU
- Convolution
- Max pooling
- Dropout
- Softmax for interpretation
- Cross-entropy loss
- Backpropagation
- Stochastic gradient descent
- Data augmentation with flipping and rotation

The notebook avoids extra analysis packages and advanced pretrained models.


In [ ]:
# Cell 1: Imports
# Only PyTorch / torchvision utilities are used for the modeling workflow.

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms


In [ ]:
# Cell 2: Settings

SEED = 617
IMAGE_SIZE = 64
BATCH_SIZE = 32

BASELINE_EPOCHS = 5
CNN_EPOCHS = 8

BASELINE_LR = 0.01
CNN_LR = 0.01

torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)


## Data folder

Put the dataset into this structure before running the next cell:

```text
data/
├── train/
│   ├── freshapples/
│   ├── freshbanana/
│   ├── freshoranges/
│   ├── rottenapples/
│   ├── rottenbanana/
│   └── rottenoranges/
└── test/
    ├── freshapples/
    ├── freshbanana/
    ├── freshoranges/
    ├── rottenapples/
    ├── rottenbanana/
    └── rottenoranges/
```

If your folder is in a different location, only change `TRAIN_DIR` and `TEST_DIR`.


In [ ]:
# Cell 3: Dataset paths

TRAIN_DIR = "data/train"
TEST_DIR = "data/test"


In [ ]:
# Cell 4: Image transformations

basic_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

augmentation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])


In [ ]:
# Cell 5: Load the image datasets

full_train_basic = datasets.ImageFolder(
    TRAIN_DIR,
    transform=basic_transform
)

full_train_augmented = datasets.ImageFolder(
    TRAIN_DIR,
    transform=augmentation_transform
)

test_data = datasets.ImageFolder(
    TEST_DIR,
    transform=basic_transform
)

class_names = full_train_basic.classes
number_of_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", number_of_classes)
print("Original training images:", len(full_train_basic))
print("Test images:", len(test_data))


In [ ]:
# Cell 6: Split the original training set into training and validation sets

train_size = int(0.80 * len(full_train_basic))
validation_size = len(full_train_basic) - train_size

generator = torch.Generator().manual_seed(SEED)

train_basic, validation_data = random_split(
    full_train_basic,
    [train_size, validation_size],
    generator=generator
)

train_indices = train_basic.indices

train_augmented = Subset(
    full_train_augmented,
    train_indices
)

print("Training images:", len(train_basic))
print("Validation images:", len(validation_data))
print("Test images:", len(test_data))


In [ ]:
# Cell 7: DataLoaders

train_loader_basic = DataLoader(
    train_basic,
    batch_size=BATCH_SIZE,
    shuffle=True
)

train_loader_augmented = DataLoader(
    train_augmented,
    batch_size=BATCH_SIZE,
    shuffle=True
)

validation_loader = DataLoader(
    validation_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Model 1: Fully Connected Neural Network Baseline

This model flattens the image pixels first, so it does not directly use the spatial structure of the image.


In [ ]:
# Cell 8: Baseline model

class BaselineModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(
            3 * IMAGE_SIZE * IMAGE_SIZE,
            128
        )

        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(
            128,
            number_of_classes
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x


baseline_model = BaselineModel().to(device)

print(baseline_model)


In [ ]:
# Cell 9: Evaluation function

def evaluate_model(model, loader, loss_function):

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_images = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = loss_function(
                outputs,
                labels
            )

            total_loss += loss.item()

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            total_correct += (
                predictions == labels
            ).sum().item()

            total_images += labels.size(0)

    average_loss = total_loss / len(loader)

    accuracy = total_correct / total_images

    return average_loss, accuracy


In [ ]:
# Cell 10: Training function

def train_model(
    model,
    train_loader,
    validation_loader,
    epochs,
    learning_rate,
    save_name
):

    loss_function = nn.CrossEntropyLoss()

    optimizer = optim.SGD(
        model.parameters(),
        lr=learning_rate
    )

    best_validation_accuracy = 0.0

    for epoch in range(epochs):

        model.train()

        total_loss = 0.0
        total_correct = 0
        total_images = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = loss_function(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            total_correct += (
                predictions == labels
            ).sum().item()

            total_images += labels.size(0)

        train_loss = total_loss / len(train_loader)
        train_accuracy = total_correct / total_images

        validation_loss, validation_accuracy = evaluate_model(
            model,
            validation_loader,
            loss_function
        )

        print(
            "Epoch",
            epoch + 1,
            "| Train Loss:",
            round(train_loss, 4),
            "| Train Accuracy:",
            round(train_accuracy, 4),
            "| Validation Loss:",
            round(validation_loss, 4),
            "| Validation Accuracy:",
            round(validation_accuracy, 4)
        )

        if validation_accuracy > best_validation_accuracy:

            best_validation_accuracy = validation_accuracy

            torch.save(
                model.state_dict(),
                save_name
            )

    model.load_state_dict(
        torch.load(
            save_name,
            map_location=device
        )
    )

    return model, best_validation_accuracy


In [ ]:
# Cell 11: Train the baseline model

baseline_model, baseline_validation_accuracy = train_model(
    baseline_model,
    train_loader_basic,
    validation_loader,
    BASELINE_EPOCHS,
    BASELINE_LR,
    "baseline_model.pth"
)

print()
print(
    "Best baseline validation accuracy:",
    round(baseline_validation_accuracy, 4)
)


# Model 2: Convolutional Neural Network

The CNN uses:
- convolutional layers
- ReLU
- max pooling
- dropout
- a final fully connected classifier


In [ ]:
# Cell 12: CNN model

class FruitCNN(nn.Module):

    def __init__(self, dropout_rate=0.3):

        super().__init__()

        self.conv1 = nn.Conv2d(
            3,
            16,
            kernel_size=3,
            padding=1
        )

        self.relu1 = nn.ReLU()

        self.pool1 = nn.MaxPool2d(
            kernel_size=2
        )

        self.conv2 = nn.Conv2d(
            16,
            32,
            kernel_size=3,
            padding=1
        )

        self.relu2 = nn.ReLU()

        self.pool2 = nn.MaxPool2d(
            kernel_size=2
        )

        self.flatten = nn.Flatten()

        self.dropout = nn.Dropout(
            dropout_rate
        )

        self.fc1 = nn.Linear(
            32 * 16 * 16,
            128
        )

        self.relu3 = nn.ReLU()

        self.fc2 = nn.Linear(
            128,
            number_of_classes
        )

    def forward(self, x):

        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)

        x = self.dropout(x)

        x = self.fc1(x)
        x = self.relu3(x)

        x = self.fc2(x)

        return x


In [ ]:
# Cell 13: Train CNN without data augmentation

cnn_model = FruitCNN(
    dropout_rate=0.3
).to(device)

cnn_model, cnn_validation_accuracy = train_model(
    cnn_model,
    train_loader_basic,
    validation_loader,
    CNN_EPOCHS,
    CNN_LR,
    "cnn_model.pth"
)

print()
print(
    "Best CNN validation accuracy:",
    round(cnn_validation_accuracy, 4)
)


# Model 3: CNN with Data Augmentation

The architecture stays the same. The training images are randomly flipped and slightly rotated.


In [ ]:
# Cell 14: Train CNN with data augmentation

cnn_augmented_model = FruitCNN(
    dropout_rate=0.3
).to(device)

cnn_augmented_model, cnn_augmented_validation_accuracy = train_model(
    cnn_augmented_model,
    train_loader_augmented,
    validation_loader,
    CNN_EPOCHS,
    CNN_LR,
    "cnn_augmented_model.pth"
)

print()
print(
    "Best CNN + augmentation validation accuracy:",
    round(cnn_augmented_validation_accuracy, 4)
)


In [ ]:
# Cell 15: Compare the models

print("Validation Accuracy")
print("-------------------")
print(
    "Baseline:",
    round(baseline_validation_accuracy, 4)
)
print(
    "CNN:",
    round(cnn_validation_accuracy, 4)
)
print(
    "CNN + Augmentation:",
    round(cnn_augmented_validation_accuracy, 4)
)


In [ ]:
# Cell 16: Select the better CNN using validation accuracy

if cnn_augmented_validation_accuracy >= cnn_validation_accuracy:

    final_model = cnn_augmented_model
    final_model_name = "CNN + Augmentation"

else:

    final_model = cnn_model
    final_model_name = "CNN"

print("Selected final model:", final_model_name)


In [ ]:
# Cell 17: Final test-set evaluation

loss_function = nn.CrossEntropyLoss()

test_loss, test_accuracy = evaluate_model(
    final_model,
    test_loader,
    loss_function
)

print("Final model:", final_model_name)
print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy, 4))


In [ ]:
# Cell 18: Confusion matrix using only PyTorch

confusion_matrix = torch.zeros(
    number_of_classes,
    number_of_classes,
    dtype=torch.int64
)

final_model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = final_model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        ).cpu()

        for true_label, predicted_label in zip(
            labels,
            predictions
        ):

            confusion_matrix[
                true_label.item(),
                predicted_label.item()
            ] += 1

print("Class order:")
print(class_names)

print()
print("Confusion matrix:")
print(confusion_matrix)


In [ ]:
# Cell 19: Per-class accuracy

print("Per-Class Accuracy")
print("------------------")

for i in range(number_of_classes):

    total_for_class = confusion_matrix[i].sum().item()

    correct_for_class = confusion_matrix[i, i].item()

    if total_for_class > 0:

        class_accuracy = (
            correct_for_class
            / total_for_class
        )

    else:

        class_accuracy = 0.0

    print(
        class_names[i],
        ":",
        round(class_accuracy, 4)
    )


In [ ]:
# Cell 20: Save the final model for the web application

checkpoint = {
    "model_state_dict": final_model.state_dict(),
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "dropout_rate": 0.3,
    "model_name": final_model_name,
    "test_accuracy": test_accuracy
}

torch.save(
    checkpoint,
    "best_model.pth"
)

print("Saved best_model.pth")


In [ ]:
# Cell 21: Convert model output to probabilities

images, labels = next(
    iter(test_loader)
)

one_image = images[0].unsqueeze(0).to(device)

final_model.eval()

with torch.no_grad():

    output = final_model(one_image)

    probabilities = torch.softmax(
        output,
        dim=1
    )

predicted_index = torch.argmax(
    probabilities,
    dim=1
).item()

print(
    "Actual class:",
    class_names[labels[0].item()]
)

print(
    "Predicted class:",
    class_names[predicted_index]
)

print(
    "Confidence:",
    round(
        probabilities[0, predicted_index].item(),
        4
    )
)
